<a href="https://colab.research.google.com/github/abod73/AI_translation/blob/main/Ai_translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══════════════════════════════════════════════════════════
# الخلية 1: تثبيت المتطلبات
# ═══════════════════════════════════════════════════════════
import os
import nest_asyncio
nest_asyncio.apply()

print("📦 جاري تثبيت المتطلبات...")
!apt-get update -qq > /dev/null
!apt-get install -y -qq ffmpeg > /dev/null 2>&1
!pip install -q --force-reinstall openai-whisper yt-dlp > /dev/null
!pip install -q transformers torch accelerate sentencepiece > /dev/null

print("✅ تم تثبيت المتطلبات بنجاح!")

# التحقق من GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("⚠️ لا يوجد GPU - سيستخدم CPU (أبطأ بكثير)")

In [ ]:
# ═══════════════════════════════════════════════════════════
# الخلية 2: تحميل الفيديو
# ═══════════════════════════════════════════════════════════
import os

# ️ ضع رابط الحلقة هنا
VIDEO_URL = "https://video.twimg.com/amplify_video/2085108324899414016/pl/cPi1B8JacKHB9gQ5.m3u8?tag=29"

print("⬇️ جاري تحميل الفيديو...")
# جودة 720p كافية جداً لـ Whisper large-v3
!yt-dlp -f "bestvideo[height<=720]+bestaudio/best[height<=720]" -o "episode.mp4" "{VIDEO_URL}"

if os.path.exists("episode.mp4"):
    size_mb = os.path.getsize("episode.mp4") / (1024 * 1024)
    print(f"✅ تم تحميل الفيديو! الحجم: {size_mb:.1f} MB")
else:
    print("❌ فشل التحميل")

In [ ]:
# ═══════════════════════════════════════════════════════════
# الخلية 3: استخراج الكلام التركي - Whisper Large-v3
# ═══════════════════════════════════════════════════════════
import os

print("🎙️ جاري استخراج الكلام التركي باستخدام Whisper large-v3...")
print(" هذا قد يستغرق 40-90 دقيقة للحلقة ساعتين (حسب GPU)")
print("🔥 لا تغلق المتصفح!")

# large-v3 = أعلى دقة ممكنة
!whisper episode.mp4 \
    --task transcribe \
    --language Turkish \
    --model large-v3 \
    --output_format srt \
    --output_dir . \
    --beam_size 5 \
    --vad_filter true \
    --condition_on_previous_text true \
    --temperature 0.0

if os.path.exists("episode.srt"):
    # عد المقاطع
    with open("episode.srt", 'r', encoding='utf-8') as f:
        content = f.read()
    num_segments = len([b for b in content.strip().split('\n\n') if b.strip()])
    print(f"✅ تم استخراج الكلام! عدد المقاطع: {num_segments}")
else:
    print("❌ فشل الاستخراج")

In [ ]:
# ═══════════════════════════════════════════════════════════
# الخلية 4: الترجمة إلى العربية - Qwen3-8B-Instruct
# ═══════════════════════════════════════════════════════════
import os, re, time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

INPUT_SRT = "episode.srt"
OUTPUT_SRT = "episode_arabic.srt"
TEMP_SRT = "episode_arabic_temp.srt"
MODEL_NAME = "Qwen/Qwen3-8B-Instruct"  # أحدث نموذج من Qwen (2025)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ═══════════════════════════════════════════════════════════
# قاموس أسماء الشخصيات (مسلسل قانون الطبيعة - Doğa Kanunu)
# ═══════════════════════════════════════════════════════════
NAMES_DICT = {
    "Doğa": "دوغا", "Yaman": "يمان", "Saffet": "سافيت",
    "Neslin": "نسلين", "Perihan": "بيران", "Hediye": "هدية",
    "Kamuran": "كمران", "Hulusi": "خلوصي", "Mert": "ميرت",
    "Burak": "بوراك", "Aytekin": "آيتكين", "Tuna": "تونا",
    "Devin": "ديفين", "Haluk": "هالوك", "Aslı": "أصلي",
}

# ═══════════════════════════════════════════════════════════
# الـ Prompt الاحترافي لـ Qwen3
# ═══════════════════════════════════════════════════════════
SYSTEM_PROMPT = """أنت مترجم محترف وخبير في دبلجة المسلسلات التركية الدرامية إلى العربية الفصحى المبسطة.

⚠️ قواعد صارمة جداً:
1. ممنوع الترجمة الحرفية - افهم المعنى واكتبه بالعربية كما سيقوله ممثل عربي.
2. لا تترجم أسماء الأشخاص (ستظهر كـ __NAME_X__).
3. التعبيرات المجازية: افهم السياق (مثلاً "الأزرق" قد يعني الاكتئاب).
4. صحح الأخطاء فوراً: 'مملوءة'→'مليئة'، 'إتفقنا'→'اتّفقنا'، 'ياصديقي'→'يا صديقي'.
5. تجنب الركاكة: 'أوشكتكم على' تعني 'جعلتكم'.
6. الجمل قصيرة ومباشرة لتناسب الترجمة.
7. أخرج الترجمة العربية فقط - بدون أي شرح أو هوامش أو علامات اقتباس."""

def build_qwen3_prompt(turkish_text, previous_context=""):
    """بناء Prompt بصيغة Qwen3 مع تعطيل thinking mode"""
    user_content = f"النص التركي: {turkish_text}"
    if previous_context:
        user_content = f"السياق السابق: {previous_context}\n\n{user_content}"
    user_content += "\n\nالترجمة العربية الطبيعية:"

    # صيغة Qwen3 - بدون thinking mode للحصول على ترجمة مباشرة
    return (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{user_content}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

# ═══════════════════════════════════════════════════════════
# حماية الأسماء
# ═══════════════════════════════════════════════════════════
def protect_names(text):
    mapping = {}
    protected = text
    for i, (tr, ar) in enumerate(NAMES_DICT.items()):
        placeholder = f"__NAME{i}__"
        pattern = re.compile(r'\b' + re.escape(tr) + r'\b', re.IGNORECASE)
        if pattern.search(protected):
            protected = pattern.sub(placeholder, protected)
            mapping[placeholder] = ar
    return protected, mapping

def restore_names(text, mapping):
    for p, ar in mapping.items():
        text = text.replace(p, ar)
    return text

def fix_errors(text):
    corrections = {
        r"\bمملوءة\b": "مليئة",
        r"\bإتفقنا\b": "اتّفقنا",
        r"\bياصديقي\b": "يا صديقي",
        r"\bأوشك(كم|تكم|ت)\s+على\b": "جعلتكم",
        r"\bاللون الأزرق\b": "الاكتئاب",
        r"\bصلوا أن\b": "ادعوا أن",
        r"\bأُصِبنا بالإصابة\b": "أُصِبنا",
    }
    for w, r in corrections.items():
        text = re.sub(w, r, text, flags=re.IGNORECASE)
    return text

# ═══════════════════════════════════════════════════════════
# تحميل نموذج Qwen3
# ═══════════════════════════════════════════════════════════
print(f"🔄 جاري تحميل نموذج {MODEL_NAME}...")
print("⏳ هذا قد يستغرق 3-8 دقائق في أول مرة...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

if DEVICE == "cuda":
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        # تعطيل thinking mode للحصول على ترجمة مباشرة
        attn_implementation="flash_attention_2" if DEVICE == "cuda" else None
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,
        trust_remote_code=True
    ).to(DEVICE)

model.eval()
print("✅ تم تحميل Qwen3 بنجاح!\n")

# ═══════════════════════════════════════════════════════════
# قراءة ملف SRT التركي
# ═══════════════════════════════════════════════════════════
print(f"📖 قراءة ملف: {INPUT_SRT}")
with open(INPUT_SRT, 'r', encoding='utf-8') as f:
    blocks = f.read().strip().split('\n\n')

segments = []
for b in blocks:
    lines = b.strip().split('\n')
    if len(lines) >= 3:
        try:
            segments.append({
                'idx': lines[0],
                'time': lines[1],
                'text': '\n'.join(lines[2:])
            })
        except:
            continue

print(f"📊 عدد المقاطع: {len(segments)}")

# ═══════════════════════════════════════════════════════════
# الاستئناف من حيث توقف (إذا انقطع Colab)
# ═══════════════════════════════════════════════════════════
start_idx = 0
translated_data = []
if os.path.exists(TEMP_SRT):
    with open(TEMP_SRT, 'r', encoding='utf-8') as f:
        temp_blocks = f.read().strip().split('\n\n')
        start_idx = len(temp_blocks)
        translated_data = temp_blocks
    print(f"⏩ استئناف من المقطع {start_idx + 1}\n")

# ═══════════════════════════════════════════════════════════
# بدء الترجمة
# ═══════════════════════════════════════════════════════════
print("🌍 بدء الترجمة بـ Qwen3...\n")
start_time = time.time()
prev_context = ""

for i in range(start_idx, len(segments)):
    seg = segments[i]
    protected_text, mapping = protect_names(seg['text'])

    prompt = build_qwen3_prompt(protected_text, prev_context)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.3,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            # تعطيل thinking mode في Qwen3
            # (Qwen3 يدعم thinking mode لكن للترجمة نريده معطلاً)
        )

    # فك التشفير - نأخذ التوكنات الجديدة فقط
    new_tokens = outputs[0][inputs.input_ids.shape[1]:]
    result = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # تنظيف النتيجة
    # إزالة أي نص تفكير إذا ظهر
    if "<think>" in result:
        result = result.split("</think>")[-1].strip()

    # إزالة "الترجمة العربية:" إذا ظهرت
    if result.startswith("الترجمة العربية:"):
        result = result[len("الترجمة العربية:"):].strip()
    if result.startswith("الترجمة:"):
        result = result[len("الترجمة:"):].strip()

    # إزالة علامات الاقتباس
    result = result.strip('"').strip("'").strip()

    # إعادة الأسماء وتصحيح الأخطاء
    final_text = fix_errors(restore_names(result, mapping))

    # تحديث السياق (آخر جملتين)
    if final_text:
        prev_context = final_text
        if translated_data:
            # استخراج آخر جملة من البيانات المترجمة
            last_block = translated_data[-1]
            last_lines = last_block.split('\n')
            if len(last_lines) >= 3:
                last_arabic = '\n'.join(last_lines[2:])
                prev_context = f"{last_arabic} | {final_text}"

    translated_data.append(f"{seg['idx']}\n{seg['time']}\n{final_text}")

    # حفظ مؤقت كل 20 مقطع
    if (i + 1) % 20 == 0:
        with open(TEMP_SRT, 'w', encoding='utf-8-sig') as f:
            f.write('\n\n'.join(translated_data) + '\n\n')

        elapsed = time.time() - start_time
        avg = elapsed / (i + 1)
        remaining = avg * (len(segments) - i - 1)
        print(f"💾 [{i+1}/{len(segments)}] | المتبقي: ~{remaining/60:.1f} دقيقة")

# ═══════════════════════════════════════════════════════════
# الحفظ النهائي
# ═══════════════════════════════════════════════════════════
with open(OUTPUT_SRT, 'w', encoding='utf-8-sig') as f:
    f.write('\n\n'.join(translated_data) + '\n\n')

if os.path.exists(TEMP_SRT):
    os.remove(TEMP_SRT)

total_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"🎉 تم الانتهاء بنجاح!")
print(f"📊 عدد المقاطع: {len(translated_data)}")
print(f"⏱️  الوقت الكلي: {total_time/60:.1f} دقيقة")
print(f" الملف المحفوظ: {OUTPUT_SRT}")
print(f"{'='*60}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# الخلية 5: تحميل ملف الترجمة العربي
# ═══════════════════════════════════════════════════════════
from google.colab import files

print("📥 جاري تحميل ملف الترجمة العربي...")
files.download("episode_arabic.srt")
print("✅ تم التحميل!")